# NeSLE Colab A100 Training Notebook

This notebook is the end-to-end A100 training and performance-test path for NeSLE. It mounts Google Drive, installs the repo, builds the CUDA extension for A100 (`sm_80`), verifies snapshot reset, benchmarks SB3 PPO against the CUDA-native PPO path, trains with checkpoints saved to Drive, and supports resume/evaluation cells.

**ROM note:** the ROM is not stored in Git. Put your legally obtained `Super Mario Bros. (World).nes` in Google Drive and set `ROM_PATH` below.


## 1. Mount Drive And Configure Paths


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

REPO_URL = 'https://github.com/hbofz/Nesle-codex.git'
BRANCH = 'main'  # set this to your native-PPO branch before running in Colab
PROJECT_DIR = Path('/content/Nesle-codex')
DRIVE_ROOT = Path('/content/drive/MyDrive/nesle')
ROM_PATH = DRIVE_ROOT / 'roms' / 'Super Mario Bros. (World).nes'
RUN_DIR = DRIVE_ROOT / 'runs' / 'a100_native_ppo'
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
MODEL_PATH = RUN_DIR / 'models' / 'nesle_ppo_curriculum'
NATIVE_PPO_CHECKPOINT = RUN_DIR / 'models' / 'nesle_native_ppo.pt'
TENSORBOARD_DIR = RUN_DIR / 'tb'
PERF_RESULTS_PATH = RUN_DIR / 'perf_results.jsonl'
# For RAM observations + MlpPolicy, SB3 is often faster on CPU even when the emulator runs on CUDA.
SB3_DEVICE = 'cpu'  # change to 'cuda' for CNN/RGB policies or to test policy-GPU speed

for path in [DRIVE_ROOT / 'roms', CHECKPOINT_DIR, MODEL_PATH.parent, TENSORBOARD_DIR, PERF_RESULTS_PATH.parent]:
    path.mkdir(parents=True, exist_ok=True)

print('ROM_PATH =', ROM_PATH)
print('RUN_DIR  =', RUN_DIR)
print('BRANCH   =', BRANCH)
print('SB3_DEVICE =', SB3_DEVICE)
print('NATIVE_PPO_CHECKPOINT =', NATIVE_PPO_CHECKPOINT)
print('PERF_RESULTS_PATH =', PERF_RESULTS_PATH)
if not ROM_PATH.exists():
    raise FileNotFoundError(f'Put your ROM at: {ROM_PATH}')


## 2. GPU And PyTorch Check


In [ ]:
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('torch cuda:', torch.version.cuda)
print('torch.cuda.is_available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')
assert torch.cuda.is_available(), 'Colab runtime is not using a CUDA GPU. Switch runtime to A100 GPU.'

x = torch.ones((512, 512), device='cuda')
torch.cuda.synchronize()
print('cuda tensor smoke:', float((x + x).sum().item()))


## 3. Clone Or Update Repo


In [ ]:
import os, subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    GITHUB_TOKEN = ''

clone_url = REPO_URL
if GITHUB_TOKEN and REPO_URL.startswith('https://github.com/'):
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@', 1)

if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, clone_url, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print('repo:', PROJECT_DIR)
!git log --oneline -5


## 4. Install Python Package And Build CUDA Extension


In [ ]:
%cd /content/Nesle-codex
!python -m pip install -U pip setuptools wheel pybind11
!python -m pip install -e '.[dev,rl]'

# Colab's running notebook kernel may not reload editable-install .pth files
# immediately. Add src explicitly for this kernel and export PYTHONPATH for
# subprocess cells.
import os, sys
os.environ['PYTHONPATH'] = f"{PROJECT_DIR / 'src'}:{os.environ.get('PYTHONPATH', '')}"
if str(PROJECT_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / 'src'))

# A100 = sm_80. H100 = sm_90.
%env NESLE_CUDA_ARCH=sm_80
!sh scripts/build_cuda_extension.sh

import importlib.util
import nesle
print('nesle:', nesle.__file__)
print('_cuda_core:', importlib.util.find_spec('nesle._cuda_core'))

from nesle import _cuda_core
assert hasattr(_cuda_core.CudaBatch, 'step_device'), 'CUDA extension is missing native PPO step_device; rebuild from the native-PPO branch.'
assert hasattr(_cuda_core.CudaBatch, 'ram_device'), 'CUDA extension is missing ram_device; rebuild from the native-PPO branch.'
print('native PPO CUDA bridge methods: ok')


## 5. CUDA-Native Tensor Bridge Smoke


In [ ]:
import torch
import nesle

BRIDGE_NUM_ENVS = 32
bridge_env = nesle.make_vec(
    rom_path=str(ROM_PATH),
    num_envs=BRIDGE_NUM_ENVS,
    frameskip=4,
    action_space='simple',
    backend='cuda',
    observation_mode='ram',
    reset_state_path='docs/data/smb_level1_1.state',
    max_episode_steps=512,
)
bridge_batch = bridge_env._cuda_batch
bridge_batch.reset_device()
ram = torch.utils.dlpack.from_dlpack(bridge_batch.ram_device())
print('ram tensor:', ram.shape, ram.dtype, ram.device)
assert ram.shape == (BRIDGE_NUM_ENVS, 2048)
assert ram.is_cuda

action_masks = torch.full((BRIDGE_NUM_ENVS,), bridge_env.action_masks[1], dtype=torch.uint8, device='cuda')
out = bridge_batch.step_device(action_masks, auto_reset=True, synchronize=True)
rewards = torch.utils.dlpack.from_dlpack(out['rewards'])
dones = torch.utils.dlpack.from_dlpack(out['dones'])
next_ram = torch.utils.dlpack.from_dlpack(out['ram'])
print('step_device:', rewards.shape, rewards.dtype, dones.shape, dones.dtype, next_ram.shape, next_ram.device)
assert rewards.is_cuda and dones.is_cuda and next_ram.is_cuda
bridge_env.close()


## 6. Snapshot Assets And Fast Verification


In [ ]:
from pathlib import Path
STATE_PATHS = [PROJECT_DIR / 'docs' / 'data' / f'smb_level{w}_1.state' for w in range(1, 9)]
for p in STATE_PATHS:
    assert p.exists(), f'missing state: {p}'
print('states:')
for p in STATE_PATHS:
    print(' ', p.name, p.stat().st_size, 'bytes')
print('rom:', ROM_PATH, ROM_PATH.stat().st_size, 'bytes')

!python -m pytest tests/test_fcs_parser.py tests/test_env_reset_state.py tests/test_env_multi_level.py tests/test_render_freshness.py -q


## 7. Optional GPU Correctness Benchmark
This can take a few minutes, but it is the best sanity check before a long run.


In [ ]:
# The benchmark scripts expect a local ROM at the repo root. Symlink/copy from Drive for this Colab session only.
repo_rom = PROJECT_DIR / 'Super Mario Bros. (World).nes'
if not repo_rom.exists():
    import shutil
    shutil.copy2(ROM_PATH, repo_rom)
!python benchmarks/verify_correctness.py
!python benchmarks/gpu_vs_cpu.py


## 8. SB3 PPO Device Speed Probe

This quick probe compares PPO with `--sb3-device cpu` and `--sb3-device cuda` while keeping NeSLE's emulator on CUDA. For RAM observations + `MlpPolicy`, CPU policy updates are often faster because SB3 stores rollouts as CPU NumPy arrays and the policy network is tiny.


In [ ]:
import json, time

PROBE_TIMESTEPS = 4096
PROBE_NUM_ENVS = 512
for probe_device in ['cpu', 'cuda']:
    print('\n=== SB3 device probe:', probe_device, '===')
    start = time.monotonic()
    !python examples/sb3_train.py "{ROM_PATH}"       --backend cuda       --sb3-device {probe_device}       --observation-mode ram       --reset-state-path docs/data/smb_level1_1.state       --action-space simple       --num-envs {PROBE_NUM_ENVS}       --timesteps {PROBE_TIMESTEPS}       --n-steps 128       --batch-size 8192       --max-episode-steps 512       --model-path /tmp/nesle_probe_{probe_device}
    elapsed = time.monotonic() - start
    result = {
        'mode': f'sb3_probe_{probe_device}',
        'num_envs': PROBE_NUM_ENVS,
        'timesteps': PROBE_TIMESTEPS,
        'elapsed_sec': elapsed,
        'timesteps_per_sec': PROBE_TIMESTEPS / max(elapsed, 1e-9),
    }
    with PERF_RESULTS_PATH.open('a') as f:
        f.write(json.dumps(result) + '\n')
    print(json.dumps(result, indent=2))


## 9. Native PPO Performance Probe


In [ ]:
import json, time

NATIVE_PROBE_TIMESTEPS = 131_072
NATIVE_PROBE_NUM_ENVS = 1024
NATIVE_PROBE_STEPS = 128
NATIVE_PROBE_BATCH = 8192
NATIVE_PROBE_CKPT = RUN_DIR / 'models' / 'native_probe.pt'

start = time.monotonic()
!python examples/native_ppo_train.py "{ROM_PATH}"   --reset-state-path docs/data/smb_level1_1.state   --action-space simple   --num-envs {NATIVE_PROBE_NUM_ENVS}   --total-timesteps {NATIVE_PROBE_TIMESTEPS}   --n-steps {NATIVE_PROBE_STEPS}   --batch-size {NATIVE_PROBE_BATCH}   --max-episode-steps 512   --checkpoint-path "{NATIVE_PROBE_CKPT}"
elapsed = time.monotonic() - start
result = {
    'mode': 'native_ppo_probe',
    'num_envs': NATIVE_PROBE_NUM_ENVS,
    'timesteps': NATIVE_PROBE_TIMESTEPS,
    'n_steps': NATIVE_PROBE_STEPS,
    'batch_size': NATIVE_PROBE_BATCH,
    'elapsed_sec': elapsed,
    'timesteps_per_sec': NATIVE_PROBE_TIMESTEPS / max(elapsed, 1e-9),
    'checkpoint': str(NATIVE_PROBE_CKPT),
}
with PERF_RESULTS_PATH.open('a') as f:
    f.write(json.dumps(result) + '\n')
print(json.dumps(result, indent=2))


## 10. SB3 Single-Level Smoke Training With Checkpoints


In [ ]:
SMOKE_TIMESTEPS = 100_000
SMOKE_NUM_ENVS = 512
SMOKE_CHECKPOINT_EVERY = 25_000

!python examples/sb3_train.py "{ROM_PATH}" \
  --backend cuda \
  --sb3-device {SB3_DEVICE} \
  --observation-mode ram \
  --reset-state-path docs/data/smb_level1_1.state \
  --action-space simple \
  --num-envs {SMOKE_NUM_ENVS} \
  --timesteps {SMOKE_TIMESTEPS} \
  --n-steps 128 \
  --batch-size 256 \
  --max-episode-steps 512 \
  --checkpoint-dir "{CHECKPOINT_DIR}" \
  --checkpoint-freq {SMOKE_CHECKPOINT_EVERY} \
  --checkpoint-prefix nesle_w1_1 \
  --tensorboard-log "{TENSORBOARD_DIR}" \
  --model-path "{MODEL_PATH}_w1_1"


## 11. Native PPO Single-Level Training


In [ ]:
NATIVE_SMOKE_TIMESTEPS = 1_000_000
NATIVE_SMOKE_NUM_ENVS = 4096

!python examples/native_ppo_train.py "{ROM_PATH}"   --reset-state-path docs/data/smb_level1_1.state   --action-space simple   --num-envs {NATIVE_SMOKE_NUM_ENVS}   --total-timesteps {NATIVE_SMOKE_TIMESTEPS}   --n-steps 128   --batch-size 8192   --max-episode-steps 512   --checkpoint-path "{NATIVE_PPO_CHECKPOINT.with_name('nesle_native_ppo_w1_1.pt')}"


## 12. SB3 Curriculum Training On All Bundled Levels


In [ ]:
CURRICULUM_TIMESTEPS = 10_000_000
CURRICULUM_NUM_ENVS = 4096
CURRICULUM_CHECKPOINT_EVERY = 250_000
STATE_ARGS = ' '.join(str(p.relative_to(PROJECT_DIR)) for p in STATE_PATHS)

!python examples/sb3_train.py "{ROM_PATH}" \
  --backend cuda \
  --sb3-device {SB3_DEVICE} \
  --observation-mode ram \
  --reset-state-paths {STATE_ARGS} \
  --action-space simple \
  --num-envs {CURRICULUM_NUM_ENVS} \
  --timesteps {CURRICULUM_TIMESTEPS} \
  --n-steps 128 \
  --batch-size 256 \
  --max-episode-steps 1024 \
  --checkpoint-dir "{CHECKPOINT_DIR}" \
  --checkpoint-freq {CURRICULUM_CHECKPOINT_EVERY} \
  --checkpoint-prefix nesle_curriculum \
  --tensorboard-log "{TENSORBOARD_DIR}" \
  --model-path "{MODEL_PATH}"


## 13. Native PPO Curriculum Training


In [ ]:
NATIVE_CURRICULUM_TIMESTEPS = 10_000_000
NATIVE_CURRICULUM_NUM_ENVS = 4096
STATE_ARGS = ' '.join(str(p.relative_to(PROJECT_DIR)) for p in STATE_PATHS)

!python examples/native_ppo_train.py "{ROM_PATH}"   --reset-state-paths {STATE_ARGS}   --action-space simple   --num-envs {NATIVE_CURRICULUM_NUM_ENVS}   --total-timesteps {NATIVE_CURRICULUM_TIMESTEPS}   --n-steps 128   --batch-size 8192   --max-episode-steps 1024   --checkpoint-path "{NATIVE_PPO_CHECKPOINT}"


## 14. Resume From Latest SB3 Checkpoint


In [ ]:
import re
def checkpoint_step(path):
    m = re.search(r'_(\d+)_steps\.zip$', path.name)
    return int(m.group(1)) if m else -1

checkpoints = sorted(CHECKPOINT_DIR.glob('*.zip'), key=checkpoint_step)
latest = checkpoints[-1] if checkpoints else None
print('latest checkpoint:', latest)
assert latest is not None, 'No checkpoints found yet.'

RESUME_TIMESTEPS = 1_000_000
STATE_ARGS = ' '.join(str(p.relative_to(PROJECT_DIR)) for p in STATE_PATHS)
!python examples/sb3_train.py "{ROM_PATH}" \
  --resume-from "{latest}" \
  --backend cuda \
  --sb3-device {SB3_DEVICE} \
  --observation-mode ram \
  --reset-state-paths {STATE_ARGS} \
  --action-space simple \
  --num-envs {CURRICULUM_NUM_ENVS} \
  --timesteps {RESUME_TIMESTEPS} \
  --checkpoint-dir "{CHECKPOINT_DIR}" \
  --checkpoint-freq {CURRICULUM_CHECKPOINT_EVERY} \
  --checkpoint-prefix nesle_curriculum_resume \
  --model-path "{MODEL_PATH}_resumed"


## 15. Evaluate A Saved SB3 Model


In [ ]:
EVAL_MODEL = str(MODEL_PATH)
!python examples/eval_smoke.py --rom "{ROM_PATH}" --model "{EVAL_MODEL}" --state docs/data/smb_level1_1.state --steps 500


## 16. Performance Results Summary


In [ ]:
import json

if PERF_RESULTS_PATH.exists():
    rows = [json.loads(line) for line in PERF_RESULTS_PATH.read_text().splitlines() if line.strip()]
    for row in rows:
        print(row)
else:
    print('No perf results written yet:', PERF_RESULTS_PATH)


## 17. TensorBoard


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{TENSORBOARD_DIR}"
